# 06. dit.py + Flow Matching: что именно учится

DiT — сеть скорости движения в латентном пространстве. Сделаем настоящий forward уменьшенной официальной архитектуры и отдельное учебное обучение velocity. Не обучаем 2B модель на ноутбуке.


### Подготовка Google Colab

Первая ячейка загружает код курса и устанавливает зависимости. Большие веса моделей она не скачивает. Для учебных опытов достаточно CPU; полная генерация в notebook 10 требует NVIDIA GPU. Если Colab попросит перезапуск сессии после установки, перезапустите и снова выполните ячейки сверху вниз.


In [ ]:
# Этот шаг нужен только в Google Colab; локально используется установленное окружение.
from pathlib import Path
import sys, subprocess
if 'google.colab' in sys.modules:
    checkout=Path('/content/generative-ai-research')
    if not checkout.exists():
        subprocess.run(['git','clone','--depth','1','https://github.com/vladislav-vasilenko/generative-ai-research.git',str(checkout)],check=True)
    course=checkout/'kandinsky-5'
    if not (course/'labkit').is_dir():
        # Пока PR не влит, берём его ветку. После merge используется main.
        subprocess.run(['git','-C',str(checkout),'fetch','--depth','1','origin','kandinsky-5-labs'],check=True)
        subprocess.run(['git','-C',str(checkout),'checkout','--detach','FETCH_HEAD'],check=True)
    course=checkout/'kandinsky-5'
    expected={'torch':'2.8.0','transformers':'4.56.2','diffusers':'0.35.1'}
    loaded={name:str(getattr(sys.modules[name],'__version__','')) for name in expected if name in sys.modules}
    subprocess.run([sys.executable,'-m','pip','install','-r',str(course/'requirements-labs.txt')],check=True)
    if any(version.split('+')[0]!=expected[name] for name,version in loaded.items()):
        raise RuntimeError('Библиотека уже была импортирована до установки. Перезапустите сессию Colab и выполните notebook с первой ячейки.')
    if str(course) not in sys.path: sys.path.insert(0,str(course))
    print('Course:',course)
    print('Git revision:',subprocess.check_output(['git','-C',str(checkout),'rev-parse','HEAD'],text=True).strip())
else:
    print('Локальный запуск: зависимости устанавливаются по README.')


### Перед началом

Запускайте ячейки сверху вниз. Для каждой операции сначала прочитайте пояснение, затем выполните код и сравните вывод с ожидаемым. Установка описана в README; сетевых загрузок в учебном режиме нет.


In [ ]:
from pathlib import Path
import sys, os
# В Colab загрузите архив KandinskyLab-colab.zip через панель Files и распакуйте
# ячейкой из README. Здесь ищем корень курса, не меняя рабочую директорию.
candidates=[Path.cwd(), *Path.cwd().parents, Path('/content/KandinskyLab'), Path('/content/generative-ai-research/kandinsky-5')]
ROOT=next((p for p in candidates if (p/'labkit').is_dir()), None)
if ROOT is None: raise RuntimeError('Сначала распакуйте архив курса; инструкции в README.')
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
os.environ.setdefault('MPLCONFIGDIR',str(ROOT/'artifacts/matplotlib'))
import torch, numpy as np, matplotlib.pyplot as plt
from labkit.common import SOURCE, REVISION, device_info, source_excerpt, release, shifted_schedule
# Маленькие эксперименты по умолчанию CPU: быстро и одинаково в Mac/Colab.
# Для отдельной проверки Metal смените LAB_DEVICE на mps.
LAB_DEVICE='cpu'
torch.manual_seed(42)
torch.set_num_threads(2)
print(device_info())


## Часть A. Что значит Flow Matching «внутри DiT»

**DiT — архитектура функции** $v_\theta(x_t,t,c)$; **Flow Matching — способ её обучения**; **Euler — алгоритм интегрирования после обучения**. В `dit.py` нет optimizer или FM loss. Сеть получает промежуточный латент, время и conditioning, а возвращает velocity. Построение training пары и loss находится в trainer, а ODE шаг — в `generation_utils.py`.

В сохранённом репозитории есть inference implementation, а полный pretraining trainer не опубликован. Ниже воспроизводим прозрачную conditional straight-path FM постановку, согласованную со знаком upstream sampler, и отдельно показываем подтверждённый execution path. Не приписываем свой uniform time sampling всей истории обучения/дистилляции Kandinsky.

### A1. Непрерывная картина: множество частиц движется по полю

Представьте облако точек noise. В каждой точке сеть задаёт стрелку; интегратор многократно двигает частицы. Совокупность траекторий преобразует одно распределение в другое. ODE описывает отдельную траекторию:

$$\frac{dx_t}{dt}=v_\theta(x_t,t,c).$$

На уровне плотности это соответствует continuity equation $\partial_t p_t+\nabla\cdot(p_t v_t)=0$: масса перераспределяется, а не исчезает. Для обычной генерации не нужно вычислять плотность или divergence; достаточно velocity forward.

### A2. Соглашение времени именно в нашем Kandinsky sampler

$x_0$ — data latent, $x_1=\epsilon$ — noise. Строим $x_t=(1-t)x_0+t\epsilon$ и target $u=dx_t/dt=\epsilon-x_0$. Training draw может идти по t в любом порядке. **Генерация идёт t=1→0**, поэтому Euler имеет отрицательное $\Delta t$.

В части FM литературы время задаётся noise→data. Это эквивалентное соглашение после $s=1-t$, но скорость меняет знак. Нельзя взять target одного соглашения и sampler другого. [Flow Matching primary paper](https://arxiv.org/abs/2210.02747).


### A3. Визуализируем path и проверяем производную

Пока одна пара data/noise. Straight path — учебная пара, не обещание, что learned sampler по неизвестному noise попадёт именно в этот data sample. Все пары смешиваются в обучении, и сеть не знает $x_0$ напрямую.


In [ ]:
x0=torch.tensor([2.,-1.]); eps=torch.tensor([-1.,2.]); ts=torch.linspace(0,1,21)
path=(1-ts[:,None])*x0+ts[:,None]*eps
plt.plot(path[:,0],path[:,1],'.-'); plt.scatter(*x0,label='data t=0'); plt.scatter(*eps,label='noise t=1')
plt.quiver(path[::5,0],path[::5,1],*[torch.full((5,),float(v)) for v in eps-x0],angles='xy',scale_units='xy',scale=5)
plt.axis('equal'); plt.legend(); plt.show()
t=0.4; h=1e-3
finite_difference=(((1-(t+h))*x0+(t+h)*eps)-((1-(t-h))*x0+(t-h)*eps))/(2*h)
assert torch.allclose(finite_difference,eps-x0,atol=1e-3)
print('dx/dt:',finite_difference,'reverse increment has negative dt')


### A4. Почему обучение не требует ODE solver

Для каждого training sample известны $x_0$ и выбранный noise. Поэтому $x_t$ и target получаются одной формулой, без 16-step sampling. Обучаем regression:

$$L_{CFM}(\theta)=\mathbb E_{x_0,\epsilon,t,c}\|v_\theta(x_t,t,c)-(\epsilon-x_0)\|^2.$$

Математический optimum squared loss в одной точке — conditional average всех подходящих targets: $v^*(x,t,c)=\mathbb E[\epsilon-x_0\mid x_t=x,t,c]$. Поэтому **условные straight paths могут пересекаться**, а learned marginal trajectories не обязаны быть прямыми. Noise-target pair не может быть восстановлена однозначно по $x_t$.


In [ ]:
# Две разные пары пересекаются в одной точке при t=0.5.
a_data=torch.tensor([-2.]); a_noise=torch.tensor([2.])
b_data=torch.tensor([2.]); b_noise=torch.tensor([-2.])
assert torch.equal((a_data+a_noise)/2,(b_data+b_noise)/2)
targets=torch.stack([a_noise-a_data,b_noise-b_data]).flatten()
predictions=torch.linspace(-5,5,101)
loss=((predictions[:,None]-targets[None,:])**2).mean(-1)
print('targets:',targets,'best shared prediction:',predictions[loss.argmin()].item())
plt.plot(predictions,loss); plt.xlabel('shared velocity'); plt.ylabel('MSE at crossing'); plt.show()


### A5. Velocity, noise prediction и clean prediction

Для выбранного linear path $x_t=x_0+t u$. Поэтому $\hat x_0=x_t-t\hat v$ и $\hat\epsilon=x_t+(1-t)\hat v$. Velocity — не изображение и не обязательно noise prediction. Это соотношения параметризаций данного path; они не означают, что неполно обученный network выдаст точный $x_0$ за один шаг.

Проверка на известной паре позволяет поймать ошибку знака независимо от нейросети. Затем source показывает: OutLayer действительно возвращает latent channels, а sampler использует их как velocity.


In [ ]:
t=torch.tensor(0.37); xt=(1-t)*x0+t*eps; u=eps-x0
assert torch.allclose(xt-t*u,x0,atol=1e-6)
assert torch.allclose(xt+(1-t)*u,eps,atol=1e-6)
print('x0 recovered:',xt-t*u,'noise recovered:',xt+(1-t)*u)


### A6. Probability path: расписание распределений, а не готовый sampler

В `lab_two.ipynb` сначала определяется $p_s(x\mid z)$, а затем ищется поле, чьи частицы имеют нужное распределение в каждый момент. Это разные объекты: path говорит **какое облако хотим видеть**, ODE говорит **как передвигать точки**.

Во вложениях $s=0$ означает noise, $s=1$ — data. Здесь сохраняем время Kandinsky $t=1-s$: data при $t=0$, noise при $t=1$, $v_t=-u_{1-t}$. Для общей Gaussian семьи

$$x_t=\alpha_t x_0+\beta_t\epsilon,\qquad p_t(x\mid x_0)=\mathcal N(\alpha_t x_0,\beta_t^2 I).$$

Учебный linear path использует $\alpha_t=1-t$, $\beta_t=t$. Во вложенной lab two есть другой допустимый путь $\alpha_s=s$, $\beta_s=\sqrt{1-s}$: одинаковые endpoints ещё не означают одинаковые промежуточные плотности или targets.

Разберём случай, который полностью решается без нейросети: $x_0\sim\mathcal N(m,a^2)$, $\epsilon\sim\mathcal N(0,1)$ независимо. Тогда

$$\mu_t=(1-t)m,\quad S_t=(1-t)^2a^2+t^2,\quad p_t=\mathcal N(\mu_t,S_t),$$
$$v_t(x)=\dot\mu_t+\frac{\dot S_t}{2S_t}(x-\mu_t),\quad \dot\mu_t=-m,\quad \dot S_t=-2(1-t)a^2+2t.$$

Последняя формула перемещает среднее и масштаб облака. Проверим, что она сохраняет вероятность: в одном измерении continuity equation — $\partial_t p_t+\partial_x(p_t v_t)=0$. Численная производная даст небольшую погрешность, а не машинный ноль.

**Источник:** `lab_two.ipynb`, ячейки 8–11, 25; `lecture_notes.pdf`, печатные стр. 16, 18–19, Theorems 9/11. Формулы выше переведены в обратное соглашение времени.


In [ ]:
# fm_ prefix отделяет переменные этой серии опытов от последующего tiny DiT.
fm_mean, fm_std = 2.0, 0.6
def fm_moments(t):
    return (1-t)*fm_mean, (1-t)**2*fm_std**2+t**2
def fm_density(x, t):
    mu, variance = fm_moments(t)
    return np.exp(-(x-mu)**2/(2*variance))/np.sqrt(2*np.pi*variance)
def fm_velocity(x, t):
    mu, variance = fm_moments(t)
    variance_dt = -2*(1-t)*fm_std**2+2*t
    return -fm_mean + 0.5*variance_dt/variance*(x-mu)

fm_grid = np.linspace(-6, 7, 3001)
fm_t, fm_h = 0.4, 1e-4
fm_dp_dt = (fm_density(fm_grid, fm_t+fm_h)-fm_density(fm_grid, fm_t-fm_h))/(2*fm_h)
fm_flux_dx = np.gradient(fm_density(fm_grid, fm_t)*fm_velocity(fm_grid, fm_t), fm_grid)
fm_residual = fm_dp_dt + fm_flux_dx
assert np.max(np.abs(fm_residual[1:-1])) < 2e-4
fm_mass = torch.trapezoid(torch.from_numpy(fm_density(fm_grid, fm_t)), torch.from_numpy(fm_grid)).item()
assert abs(fm_mass-1) < 1e-5
print('integrated probability:', fm_mass, 'continuity residual:', np.max(np.abs(fm_residual[1:-1])))
fig, fm_axes = plt.subplots(1,2,figsize=(11,3.4))
for fm_tt in (0, 0.25, 0.5, 0.75, 1):
    fm_axes[0].plot(fm_grid, fm_density(fm_grid, fm_tt), label=f't={fm_tt:g}')
fm_axes[0].set(xlabel='x', ylabel='p_t(x)', title='Data t=0 → noise t=1'); fm_axes[0].legend()
fm_axes[1].plot(fm_grid, fm_dp_dt, label='∂t p'); fm_axes[1].plot(fm_grid, -fm_flux_dx, '--', label='−∂x(pv)')
fm_axes[1].set(xlabel='x', title='Probability conservation'); fm_axes[1].legend(); plt.tight_layout(); plt.show()


### A7. «Условная» скорость и conditioning текста — два разных условия

В conditional FM слово «условная» относится к известному training endpoint $x_0$: при $t>0$ поле $v_t(x\mid x_0)=(x-x_0)/t$ ведёт к этому конкретному примеру при движении назад. В T2V сеть получает текст $c$, но **не получает истинный endpoint** будущего видео. На inference нужны все совместимые endpoints, взвешенные по posterior:

$$v_t(x)=\mathbb E[v_t(x\mid x_0)\mid x_t=x].$$

Усреднение по исходному $p_{data}(x_0)$ неверно: уже наблюдаемое $x_t=x$ меняет наши представления о $x_0$. Для Gaussian случая Bayes даёт

$$\mathbb E[x_0\mid x_t=x]=m+\frac{(1-t)a^2}{S_t}(x-\mu_t),\qquad
\mathrm{Var}(x_0\mid x_t=x)=\frac{a^2t^2}{S_t}.$$

Подставим posterior mean в $(x-x_0)/t$. Получится **то же** marginal поле из A6. Поэтому обучающая задача может использовать простые conditional targets, хотя inference работает с неизвестным сложным marginal полем. При text conditioning аналогичная posterior усреднённость сохраняется внутри распределения видео, совместимых с $c$.

**Источник:** `lab_two.ipynb`, ячейка 39; `lecture_notes.pdf`, стр. 16/18, posterior interpretation Theorem 9. Здесь Gaussian prior выбран только для аналитической проверки.


In [ ]:
def fm_posterior_mean(x, t):
    mu, variance = fm_moments(t)
    return fm_mean + (1-t)*fm_std**2/variance*(x-mu)
fm_xcheck = np.linspace(-2,4,100)
fm_bayes_velocity = (fm_xcheck-fm_posterior_mean(fm_xcheck,fm_t))/fm_t
assert np.allclose(fm_bayes_velocity,fm_velocity(fm_xcheck,fm_t),atol=1e-12)
plt.figure(figsize=(7,3.4))
plt.plot(fm_xcheck,fm_bayes_velocity,label='posterior average = true marginal')
plt.plot(fm_xcheck,(fm_xcheck-fm_mean)/fm_t,'--',label='prior average: incorrect')
plt.axhline(0,color='gray',lw=0.6); plt.xlabel('observed x_t'); plt.ylabel('velocity at t=0.4')
plt.legend(); plt.tight_layout(); plt.show()
print('Bayes and analytic velocity agree to:', np.max(np.abs(fm_bayes_velocity-fm_velocity(fm_xcheck,fm_t))))


### A8. Почему правильный CFM loss обычно не станет нулём

Даже точная сеть не угадает случайную пару data/noise, если несколько пар приводят к одному $x_t$. Для scalar prediction $q$ и случайного target $U=\epsilon-x_0$ работает разложение

$$\mathbb E[(q-U)^2\mid x_t=x]=(q-v_t(x))^2+\mathrm{Var}(U\mid x_t=x).$$

Вторая часть не зависит от prediction: это неопределённость пары, а не ошибка сети. В Gaussian примере $\mathrm{Var}(U\mid x_t=x)=a^2/S_t$. Значит оптимум loss имеет положительный уровень. Ниже рисуем endpoints из **posterior при фиксированном наблюдении**, восстанавливаем соответствующий noise и сравниваем empirical MSE с формулой.

Это объясняет, почему одинаково ненулевой training loss может сочетаться с правильным transport, и почему сам loss не измеряет качество видео. Последующее tiny DiT обучение на одном data latent — особый случай с гораздо меньшей неоднозначностью.

**Источник:** `lecture_notes.pdf`, стр. 20–22, Theorem 12 (FM/CFM отличаются константой); `lab_two.ipynb`, ячейка 39. Разложение и Gaussian эксперимент здесь выписаны самостоятельно.


In [ ]:
fm_rng = np.random.default_rng(1606)
fm_observed = 1.2
_, fm_var = fm_moments(fm_t)
fm_post_mean = fm_posterior_mean(fm_observed,fm_t)
fm_post_var = fm_std**2*fm_t**2/fm_var
fm_endpoint_draws = fm_post_mean+np.sqrt(fm_post_var)*fm_rng.normal(size=30000)
fm_noise_draws = (fm_observed-(1-fm_t)*fm_endpoint_draws)/fm_t
fm_targets = fm_noise_draws-fm_endpoint_draws
fm_optimal = fm_velocity(fm_observed,fm_t)
fm_candidates = np.linspace(fm_optimal-2,fm_optimal+2,101)
fm_empirical_mse = ((fm_candidates[:,None]-fm_targets[None,:])**2).mean(1)
fm_irreducible = fm_std**2/fm_var
fm_theory_mse = (fm_candidates-fm_optimal)**2+fm_irreducible
assert abs(fm_targets.mean()-fm_optimal) < 0.03
assert abs(np.mean((fm_targets-fm_optimal)**2)-fm_irreducible) < 0.04
plt.plot(fm_candidates,fm_empirical_mse,label='posterior Monte Carlo MSE')
plt.plot(fm_candidates,fm_theory_mse,'--',label='(q−v)² + conditional variance')
plt.axvline(fm_optimal,color='gray',lw=0.8); plt.xlabel('prediction q'); plt.ylabel('conditional MSE')
plt.legend(); plt.show(); print('best velocity:',fm_optimal,'irreducible MSE:',fm_irreducible)


### A9. FM, score matching и DDPM: похожий input, разные targets

Score $s_t(x)=\nabla_x\log p_t(x)$ — градиент **логарифма** плотности. Он показывает локальное направление роста вероятности, а не velocity. Для Gaussian conditional path

$$s_t(x\mid x_0)=\frac{\alpha_t x_0-x}{\beta_t^2}=-\frac{\epsilon}{\beta_t}.$$

В `lab_two.ipynb`, ячейке 33, запись $\nabla\log\mathcal N=\nabla\mathcal N$ содержит опечатку: справа должно быть $\nabla\mathcal N/\mathcal N$. Правильную формулу выше подтверждает PDF, стр. 25/31.

| Задача | Учебный target на известной паре | Что нужно помнить |
|---|---|---|
| Linear-path CFM | $\epsilon-x_0$ | Скорость относится к data→noise времени; generation идёт обратно |
| Conditional score matching | $-\epsilon/\beta_t$ | Target растёт при $\beta_t\to0$; raw score MSE сильно зависит от времени |
| Noise prediction, DDPM-style objective | $\epsilon$ | Score получается как $-\hat\epsilon/\beta_t$; удаление $1/\beta_t^2$ меняет weighting loss |

Нельзя менять target и оставлять старый sampler без преобразования выхода. Эти Gaussian параметризации связаны алгеброй, но при конечной сети/time weighting не обязаны давать одинаковый результат обучения. Название **Diffusion Transformer** само по себе не означает DDPM noise prediction: attached lab three и Kandinsky используют DiT для velocity.

Для общей семьи при $\alpha_t>0$ и $\beta_t>0$:

$$v_t(x)=\frac{\dot\alpha_t}{\alpha_t}x+
\left(\beta_t^2\frac{\dot\alpha_t}{\alpha_t}-\beta_t\dot\beta_t\right)s_t(x).$$

При нашем linear path это $v_t=-(x+t s_t)/(1-t)$, когда $0<t<1$. Формула имеет деление на $1-t$, поэтому её **не вычисляем на noise endpoint**. Прямой velocity predictor Kandinsky не требует такой потенциально неудобной конверсии.

**Источник:** `lecture_notes.pdf`, стр. 26, Proposition 1; стр. 31–32, Example 23 и DDPM loss; `lab_two.ipynb`, ячейки 33/39; `lab_three.ipynb`, ячейки 24–26, 48. PDF даёт DDPM сравнение; lab three обучает именно FM, не DDPM.


In [ ]:
fm_tscore = 0.35
fm_mu, fm_variance = fm_moments(fm_tscore)
fm_score = -(fm_xcheck-fm_mu)/fm_variance
fm_from_score = -(fm_xcheck+fm_tscore*fm_score)/(1-fm_tscore)
assert np.allclose(fm_from_score,fm_velocity(fm_xcheck,fm_tscore),atol=1e-12)
# Noise predictor optimum тоже есть conditional mean, а не исходный random ε.
fm_noise_mean = -fm_tscore*fm_score
fm_data_mean = fm_xcheck-fm_tscore*fm_from_score
assert np.allclose(fm_data_mean,fm_posterior_mean(fm_xcheck,fm_tscore))
assert np.allclose(fm_noise_mean,fm_xcheck+(1-fm_tscore)*fm_from_score)
# Один fixed prediction error у ε соответствует time-dependent error score MSE.
fm_levels = np.array([0.05,0.2,0.5,0.9])
fm_noise_error = 0.1
print('t       noise MSE      equivalent raw score MSE')
for fm_tt in fm_levels:
    print(f'{fm_tt:0.2f}      {fm_noise_error**2:0.4f}            {(fm_noise_error/fm_tt)**2:0.4f}')
print('velocity / score / denoiser / noise conversions agree away from singular endpoints.')


### A10. Почему marginal trajectory изгибается и Euler требует несколько шагов

Conditional straight-path target постоянен на **одной известной паре**. Marginal поле каждый момент заново усредняет пары, совместимые с текущей точкой; оно обычно меняется вдоль траектории. Для независимых Gaussian координат точный marginal trajectory из noise $\eta$ имеет вид

$$x(t)=\mu_t+\sqrt{S_t}\,\eta.$$

Проверим это на 2D облаке с разными variance по осям. Сравним точный путь и reverse Euler $x\leftarrow x+\Delta t\,v_t(x)$ на 4–128 шагах. На этом гладком поле глобальная ошибка Euler убывает примерно как $1/N$. Путь выбирается аналитически; мы пока не измеряем ошибку neural network.

Таким образом, **16 steps у distilled Kandinsky — свойство совместного выбора обученных weights и sampler**, а не следствие того, что любой FM model точен за 16 шагов. Distillation и model error обсуждаются ниже отдельно; этот опыт изолирует только numerical integration error.

**Источник:** `lab_one.ipynb`, ячейки 7, 11 (Euler); `lab_two.ipynb`, ячейки 25/27–29 (conditional ODE). Gaussian marginal решение здесь — проверяемый аналитический пример.


In [ ]:
fm_mean2 = np.array([2.0,-1.0]); fm_std2 = np.array([0.6,1.2])
fm_initial2 = np.array([-1.0,1.5])
def fm_velocity2(x,t):
    mu=(1-t)*fm_mean2; variance=(1-t)**2*fm_std2**2+t**2
    variance_dt=-2*(1-t)*fm_std2**2+2*t
    return -fm_mean2+0.5*variance_dt/variance*(x-mu)
fm_exact_ts = np.linspace(1,0,400)
fm_exact_path = ((1-fm_exact_ts[:,None])*fm_mean2
                 +np.sqrt((1-fm_exact_ts[:,None])**2*fm_std2**2+fm_exact_ts[:,None]**2)*fm_initial2)
fm_exact_endpoint = fm_mean2+fm_std2*fm_initial2
fm_counts = np.array([4,8,16,32,64,128]); fm_errors=[]
fig, fm_axes = plt.subplots(1,2,figsize=(11,3.6))
fm_axes[0].plot(*fm_exact_path.T,'k',label='exact marginal curve')
for fm_n in fm_counts:
    fm_state=fm_initial2.copy(); fm_trace=[fm_state.copy()]
    for fm_tleft in np.linspace(1,0,fm_n+1)[:-1]:
        fm_state=fm_state-fm_velocity2(fm_state,fm_tleft)/fm_n
        fm_trace.append(fm_state.copy())
    fm_errors.append(np.linalg.norm(fm_state-fm_exact_endpoint))
    if fm_n in (4,16): fm_axes[0].plot(*np.array(fm_trace).T,'.--',label=f'Euler {fm_n} steps')
assert fm_errors[-1] < fm_errors[0]/10
fm_axes[0].set(xlabel='coordinate 1',ylabel='coordinate 2'); fm_axes[0].legend()
fm_axes[1].loglog(fm_counts,fm_errors,'o-',label='endpoint error')
fm_axes[1].loglog(fm_counts,fm_errors[0]*fm_counts[0]/fm_counts,'--',label='reference 1/N')
fm_axes[1].set(xlabel='velocity evaluations N',ylabel='Euclidean error'); fm_axes[1].legend()
plt.tight_layout(); plt.show()
print(dict(zip(fm_counts.tolist(),np.round(fm_errors,6).tolist())))


### A11. Optional SDE extension: noise требует компенсации в drift

Вложенная lab two показывает, что одному probability path могут соответствовать deterministic и stochastic trajectories. Это **теоретическое расширение**, а опубликованный Kandinsky sampler ниже делает ODE/Euler update без новых Brownian noise increments.

Чтобы не перепутать знак stochastic времени, здесь явно вводим generation clock $s=1-t$, $U_s(x)=-v_{1-s}(x)$ и интегрируем **вперёд по $s$**. Для diffusion coefficient $g_s$, зависящего только от времени, SDE с теми же marginal distributions имеет вид

$$dX_s=\left[U_s(X_s)+\tfrac12 g_s^2\nabla\log p_{1-s}(X_s)\right]ds+g_s\,dW_s.$$

Обозначим плотность в generation clock как $\widetilde p_s=p_{1-s}$. Score correction притягивает облако так, чтобы компенсировать расширение от noise. В Fokker–Planck equation появляются два взаимно сокращающихся члена: $-\tfrac12 g_s^2\partial_x(\widetilde p_s\partial_x\log\widetilde p_s)$ и $+\tfrac12 g_s^2\partial_x^2\widetilde p_s$. Просто добавить noise к прежнему Euler update — другая динамика.

Ниже сравниваем ODE, правильно скорректированную SDE и SDE без correction на том же Gaussian path. Euler–Maruyama добавляет $g\sqrt{\Delta s}\,\xi$, **не** $g\Delta s\,\xi$. Равенство marginal distributions точное в continuous theory; конечные шаги и Monte Carlo дают небольшую погрешность. Такой эксперимент не обещает ускорить или улучшить видео Kandinsky.

**Источник:** `lab_one.ipynb`, ячейки 7/12 (Euler–Maruyama); `lab_two.ipynb`, ячейки 34–35; `lecture_notes.pdf`, стр. 28–29, Theorems 17/19. Используется marginal score с положительной target variance, поэтому singular conditional endpoint из lab two здесь отсутствует.


In [ ]:
fm_sde_rng = np.random.default_rng(2026)
fm_samples, fm_sde_steps, fm_diffusion = 8000, 400, 0.7
fm_initial_cloud = fm_sde_rng.normal(size=fm_samples)
fm_initial_cloud=(fm_initial_cloud-fm_initial_cloud.mean())/fm_initial_cloud.std()
fm_ode_cloud=fm_initial_cloud.copy(); fm_sde_cloud=fm_initial_cloud.copy(); fm_naive_cloud=fm_initial_cloud.copy()
fm_ds=1/fm_sde_steps
for fm_s in np.linspace(0,1,fm_sde_steps+1)[:-1]:
    fm_tt=1-fm_s; fm_mu,fm_var=fm_moments(fm_tt)
    fm_dW=np.sqrt(fm_ds)*fm_sde_rng.normal(size=fm_samples)
    fm_score_now=-(fm_sde_cloud-fm_mu)/fm_var
    fm_ode_cloud += -fm_velocity(fm_ode_cloud,fm_tt)*fm_ds
    fm_sde_cloud += (-fm_velocity(fm_sde_cloud,fm_tt)+0.5*fm_diffusion**2*fm_score_now)*fm_ds+fm_diffusion*fm_dW
    fm_naive_cloud += -fm_velocity(fm_naive_cloud,fm_tt)*fm_ds+fm_diffusion*fm_dW
fm_clouds={'ODE':fm_ode_cloud,'SDE + score correction':fm_sde_cloud,'SDE without correction':fm_naive_cloud}
for fm_name,fm_cloud in fm_clouds.items():
    print(f'{fm_name:24s}: mean={fm_cloud.mean():.3f}, variance={fm_cloud.var():.3f}')
    plt.hist(fm_cloud,bins=65,density=True,histtype='step',lw=1.4,label=fm_name)
plt.plot(fm_grid,fm_density(fm_grid,0),'k--',label='target N(2,0.36)'); plt.xlim(-1,5)
plt.xlabel('generated coordinate'); plt.ylabel('density'); plt.legend(); plt.show()
assert abs(fm_sde_cloud.mean()-fm_mean) < 0.05
assert abs(fm_sde_cloud.var()-fm_std**2) < 0.05
assert fm_naive_cloud.var() > fm_sde_cloud.var()+0.1


### A12. От учебных class labels к Qwen/CLIP conditioning и CFG

`lab_three.ipynb` строит полный маршрут: time encoder → patchifier → Transformer blocks с modulation → depatchifier → velocity; затем class-conditional FM, VAE encoder и FM в latent space. Мы сохраняем эту логику, но используем маленькие CPU опыты и проверяем реальные классы Kandinsky ниже.

| Учебная lab three | Kandinsky 5.0 Lite | Что учит сеть |
|---|---|---|
| MNIST class embedding | Qwen token states + pooled CLIP embedding, затем trainable projections/LTF | Как conditioning меняет velocity; tokenizer сам velocity не предсказывает |
| Fourier time encoder | TimeEmbedding + modulation каждого visual block | Как интерпретировать уровень смеси data/noise |
| Image patchifier/depatchifier | Projection пространственных patches video latent + OutLayer | Как перевести latent grid в tokens и вернуть velocity channels |
| Latent FM с frozen учебным VAE | Предобученный video VAE codec и DiT | Transport в уже выбранном latent space |

CFG комбинирует **две velocity prediction**, $v_{CFG}=v_{uncond}+w(v_{cond}-v_{uncond})$. При $w=1$ сохраняется conditional ветка, то есть текст продолжает влиять на результат. В опубликованном `get_velocity` для этого значения вторая ветка пропускается. У distilled 5s `guidance=1` уменьшает число DiT forwards на шаг; это не «генерация без текста».

Отдельный опыт: две Gaussian категории, для каждой аналитически известен conditional marginal velocity; unconditional поле — posterior mixture этих категорий. Посмотрим, как CFG меняет поле. Значение $w>1$ меняет динамику и в общем случае не гарантирует лучшее качество или точное sampling исходного $p(x\mid c)$; это inference hyperparameter.

**Источник:** `lab_three.ipynb`, ячейки 24–26 (CFG), 38–48 (DiT), 75–78 (latent FM); Kandinsky `kandinsky/models/dit.py`, `kandinsky/generation_utils.py:get_velocity`. Различия в архитектуре видны в следующей части B.


In [ ]:
fm_cfg_t = 0.6
fm_class_means=np.array([-2.,2.]); fm_cfg_var=(1-fm_cfg_t)**2*fm_std**2+fm_cfg_t**2
fm_cfg_mu=(1-fm_cfg_t)*fm_class_means
fm_cfg_x=np.linspace(-3,3,250)
fm_logits=-(fm_cfg_x[:,None]-fm_cfg_mu[None,:])**2/(2*fm_cfg_var)
fm_weights=np.exp(fm_logits-fm_logits.max(1,keepdims=True)); fm_weights/=fm_weights.sum(1,keepdims=True)
fm_cfg_rate=(-(1-fm_cfg_t)*fm_std**2+fm_cfg_t)/fm_cfg_var
fm_class_vel=-fm_class_means[None,:]+fm_cfg_rate*(fm_cfg_x[:,None]-fm_cfg_mu[None,:])
fm_unconditional=(fm_weights*fm_class_vel).sum(1); fm_conditional=fm_class_vel[:,1]
for fm_w in (0.,1.,3.):
    fm_guided=fm_unconditional+fm_w*(fm_conditional-fm_unconditional)
    if fm_w==1.: assert np.allclose(fm_guided,fm_conditional)
    plt.plot(fm_cfg_x,fm_guided,label=f'guidance w={fm_w:g}')
plt.xlabel('x_t'); plt.ylabel('data→noise velocity'); plt.title('Conditioning on class with data mean +2')
plt.legend(); plt.show()
source_excerpt('kandinsky/generation_utils.py','if abs(guidance_weight - 1.0) > 1e-6:',18)


**Навигация по источникам:** [карта материалов MIT и уточнения формул](../reference/course_materials/README.md). Номера страниц относятся к предоставленному lecture_notes.pdf; номера ячеек исходных lab notebooks считаются с нуля.

### Упражнения к дополнению

1. Подставьте путь источника alpha_s=s, beta_s=sqrt(1-s) в формулу conditional velocity. Почему derivative около s=1 требует осторожности?
2. Измените std данных Gaussian примера. Сравните минимальный CFM loss и ошибку Euler: измеряют ли они одну и ту же величину?
3. В SDE опыте уменьшите шаг и сравните среднее/variance с target. Объясните, почему Brownian increment имеет масштаб sqrt(ds), а drift — ds.
4. В CFG опыте сравните w=0,1,3. Найдите в настоящем get_velocity условие второй ветки и посчитайте DiT forwards для 16 шагов при w=1 и w=3.


## Часть B. Базовые опыты и связь с исходниками

### 1. Классы dit.py

TransformerEncoderBlock уточняет текст через self-attention и FFN. TransformerDecoderBlock обрабатывает visual tokens через self-attention → cross-attention → FFN. DiffusionTransformer3D связывает embeddings, RoPE, блоки и OutLayer. get_dit — factory. В README дан перечень внутренних trainable весов.


In [ ]:
source_excerpt('kandinsky/models/dit.py','class TransformerDecoderBlock',43)


### 2. Настоящая архитектура, маленькие размеры

Уменьшаем ширину и количество блоков, но сохраняем те же классы. На вход 33 канала: 16 шумовых, 16 нулевых visual conditioning, 1 нулевая маска T2V. На выход 16 каналов velocity. Такой forward позволяет проверить contract без больших весов.


In [ ]:
from labkit.native import tiny_dit
model=tiny_dit(LAB_DEVICE)
x=torch.randn(1,4,4,16,device=LAB_DEVICE)
input33=torch.cat([x,torch.zeros_like(x),torch.zeros_like(x[...,:1])],-1)
text=torch.randn(1,6,32,device=LAB_DEVICE); pooled=torch.randn(1,16,device=LAB_DEVICE)
pos=[torch.arange(1,device=LAB_DEVICE),torch.arange(2,device=LAB_DEVICE),torch.arange(2,device=LAB_DEVICE)]
with torch.no_grad(): velocity=model(input33,text,pooled,torch.tensor([500.],device=LAB_DEVICE),pos,torch.arange(6,device=LAB_DEVICE))
print('parameters:',sum(p.numel() for p in model.parameters()),'input:',input33.shape,'output:',velocity.shape)


### 3. Увидим промежуточные формы

Forward hooks наблюдают выход, не меняя модель. Сначала visual projection выдаёт решётку [T,H/2,W/2,D]; перед блоками она превращается в [1,N,D]. OutLayer возвращает [T,H,W,16].


In [ ]:
shapes={}
def hook(name):
    def record(module,args,out):
        if isinstance(out,torch.Tensor): shapes[name]=tuple(out.shape)
    return record
handles=[module.register_forward_hook(hook(name)) for name,module in model.named_modules() if name in ['visual_embeddings','text_embeddings','visual_transformer_blocks.0','out_layer']]
with torch.no_grad(): model(input33,text,pooled,torch.tensor([500.],device=LAB_DEVICE),pos,torch.arange(6,device=LAB_DEVICE))
for h in handles: h.remove()
print(shapes)


### 4. Flow Matching: учим направление, а не картинку

Выбираем convention, совпадающий с decreasing-time sampler: x₀ — data, ε — noise, xₜ=(1−t)x₀+tε. Производная по t равна ε−x₀. Учим сеть предсказывать эту velocity. При sampling идём t:1→0, поэтому Δt отрицательно. Маленький пример доказывает знак.


In [ ]:
data=torch.tensor([2.,-1.]); noise=torch.tensor([-0.5,0.4]); t=0.7
xt=(1-t)*data+t*noise; target=noise-data
print('mixed:',xt,'velocity:',target)
print('one exact step noise→data:',noise-target)
assert torch.allclose(noise-target,data)


### 5. Учебное обучение того же tiny DiT

Возьмём единственный фиксированный data latent. Каждый шаг рисует новые noise и t. Это намеренно простая задача: уменьшение loss показывает, что градиенты доходят до DiT, но не доказывает освоение изображений или текста. Frozen conditioning ниже — сохранённые случайные векторы, а не Qwen.


In [ ]:
model.train(); optimizer=torch.optim.Adam(model.parameters(),lr=0.002)
data_latent=torch.ones_like(x)*0.3
losses=[]
for step in range(100):
    t=torch.rand(1,device=LAB_DEVICE); noise=torch.randn_like(x)
    xt=(1-t)*data_latent+t*noise
    input33=torch.cat([xt,torch.zeros_like(xt),torch.zeros_like(xt[...,:1])],-1)
    prediction=model(input33,text.detach(),pooled.detach(),t*1000,pos,torch.arange(6,device=LAB_DEVICE))
    target=noise-data_latent
    loss=torch.nn.functional.mse_loss(prediction,target)
    optimizer.zero_grad(); loss.backward(); optimizer.step(); losses.append(loss.item())
plt.plot(losses); plt.ylabel('velocity MSE'); plt.xlabel('step'); plt.show()
print('first 10 mean:',np.mean(losses[:10]),'last 10 mean:',np.mean(losses[-10:]))


### 6. Проверяем loss на новых noise/t

Используем 20 новых пар, не делая optimizer.step. Обобщение здесь ограничено одним data latent. Следующий уровень — набор латентов и осмысленный conditioning; промышленное обучение требует данных и compute, которых этот курс не воспроизводит.


In [ ]:
model.eval(); validation=[]
with torch.no_grad():
    for _ in range(20):
        t=torch.rand(1,device=LAB_DEVICE); eps=torch.randn_like(x); xt=(1-t)*data_latent+t*eps
        inp=torch.cat([xt,torch.zeros_like(xt),torch.zeros_like(xt[...,:1])],-1)
        pred=model(inp,text,pooled,t*1000,pos,torch.arange(6,device=LAB_DEVICE))
        validation.append(torch.nn.functional.mse_loss(pred,eps-data_latent).item())
print('validation MSE:',np.mean(validation))


### 7. Структура обучаемых весов

Qwen hidden states frozen, но text_embeddings и Linguistic Token Refiner внутри DiT обучаются. Равным образом CLIP frozen, но pooled_text_embeddings обучается. RoPE частоты — buffers, не параметры. Эта граница особенно важна при LoRA.


In [ ]:
groups={}
for name,p in model.named_parameters():
    group=name.split('.')[0]; groups[group]=groups.get(group,0)+p.numel()
print(groups)
print('buffers:',[(n,tuple(b.shape)) for n,b in model.named_buffers()])


#


In [ ]:
#


## Часть C. Разложим настоящий Kandinsky forward на этапы

Выполним **те же методы**, что вызывает `DiffusionTransformer3D.forward`:

1. Qwen hidden states → Linear/LayerNorm; CLIP pooled → projection; t → sinusoidal/MLP; time+CLIP складываются.
2. Visual latent → patches; text → 1D RoPE и Linguistic Token Refiner.
3. Visual grid → flatten + 3D RoPE → CrossDiT blocks.
4. Modulated OutLayer → unpatchify → 16-channel velocity.

Время ODE t измеряется от 0 до 1, но network получает **t×1000** — это отдельный contract embeddings, не 1000 sampling steps. В input T2V есть дополнительные zero visual channels/mask. На маленьком примере сравним ручное исполнение stages с единым forward.


In [ ]:
from labkit.native import tiny_dit
trace_model=tiny_dit('cpu')
trace_x=torch.randn(1,4,4,33); trace_text=torch.randn(1,6,32); trace_clip=torch.randn(1,16)
trace_t=torch.tensor([0.5])*1000
trace_pos=[torch.arange(1),torch.arange(2),torch.arange(2)]; text_pos=torch.arange(6)
with torch.no_grad():
    te,time_emb,text_rope,ve=trace_model.before_text_transformer_blocks(trace_text,trace_t,trace_clip,trace_x,text_pos)
    print('projected text:',te.shape,'time+CLIP:',time_emb.shape,'visual grid:',ve.shape)
    for block in trace_model.text_transformer_blocks: te=block(te,time_emb,text_rope)
    ve,visual_shape,to_fractal,visual_rope=trace_model.before_visual_transformer_blocks(ve,trace_pos,(1,1,1),None)
    print('visual sequence:',ve.shape,'3D RoPE:',visual_rope.shape)
    for block in trace_model.visual_transformer_blocks: ve=block(ve,te,time_emb,visual_rope,None)
    staged=trace_model.after_blocks(ve,visual_shape,to_fractal,te,time_emb)
    direct=trace_model(trace_x,trace_text,trace_clip,trace_t,trace_pos,text_pos)
assert torch.allclose(staged,direct,atol=1e-6)
print('velocity:',direct.shape,'stage/forward difference:',(staged-direct).abs().max().item())
source_excerpt('kandinsky/models/dit.py','text_embed, time_embed, text_rope, visual_embed = self.before_text_transformer_blocks',18)


### C2. Как время и текст меняют velocity

Cross-attention вводит детальные Qwen tokens; CLIP pooled меняет modulation вместе со временем. В fresh модели modulation gates zero-initialized: некоторые residual ветви сначала могут не влиять на результат. После обучения dependence проверяется экспериментом, а не предполагается по названию attention.

На tiny DiT, обученном ранее для одного фиксированного latent, сравним ответы при разных t. Различие outputs показывает dependence; оно не доказывает точность и не показывает семантическое понимание Qwen. Случайный conditioning в учебном training не эквивалентен настоящим embeddings.


In [ ]:
probe=torch.randn_like(x)
probe_input=torch.cat([probe,torch.zeros_like(probe),torch.zeros_like(probe[...,:1])],-1)
with torch.no_grad():
    v_early=model(probe_input,text,pooled,torch.tensor([900.],device=LAB_DEVICE),pos,torch.arange(6,device=LAB_DEVICE))
    v_late=model(probe_input,text,pooled,torch.tensor([100.],device=LAB_DEVICE),pos,torch.arange(6,device=LAB_DEVICE))
print('mean |velocity(t=.9)-velocity(t=.1)|:',(v_early-v_late).abs().mean().item())


## Часть D. Полный маленький conditional flow: учим два распределения

Дополнительная 2D задача отделяет статистический смысл FM от сложности video Transformer. Вместо DiT используем маленький MLP, но path, target и solver одинакового типа. Conditioning class=0/1 играет роль двух разных промптов. Data distribution — Gaussian clouds вокруг двух центров, noise — один стандартный Gaussian.

Обучение: случайные data/noise/t → один network forward → MSE → update. Генерация: только noise + выбранный class → повторные velocity forwards; data sample на вход не подаётся. Это принципиальное различие с реконструкцией VAE.


In [ ]:
class ToyVelocity(torch.nn.Module):
    def __init__(self):
        super().__init__(); self.net=torch.nn.Sequential(torch.nn.Linear(5,64),torch.nn.SiLU(),torch.nn.Linear(64,64),torch.nn.SiLU(),torch.nn.Linear(64,2))
    def forward(self,x,t,c):
        return self.net(torch.cat([x,t,torch.nn.functional.one_hot(c,2).float()],-1))
torch.manual_seed(12)
field=ToyVelocity(); optimizer=torch.optim.Adam(field.parameters(),lr=0.003)
centers=torch.tensor([[-2.,-0.7],[2.,0.7]]); flow_losses=[]
for step in range(650):
    classes=torch.randint(0,2,(128,)); data2=centers[classes]+0.3*torch.randn(128,2)
    noise2=torch.randn_like(data2); times=torch.rand(128,1)
    mixed=(1-times)*data2+times*noise2
    loss=torch.nn.functional.mse_loss(field(mixed,times,classes),noise2-data2)
    optimizer.zero_grad(); loss.backward(); optimizer.step(); flow_losses.append(loss.item())
plt.plot(flow_losses); plt.xlabel('training step'); plt.ylabel('CFM velocity MSE'); plt.show()


### D2. Генерируем без encoder и без data samples

Для каждого class используем одинаковый initial noise: так разница результата зависит от conditioning. Сначала сохраняем intermediate trajectory, потом сравниваем final samples с известными training distribution centers. Ошибка mean и spread — учебные проверки; они не являются image/video quality metrics.

Euler идёт по decreasing t. Uniform grid здесь выбран для ясности; Kandinsky distill grid исследуем следующим разделом. Даже если training paths прямые, sampled trajectory learned field может изгибаться.


In [ ]:
field.eval(); initial=torch.randn(300,2); generated={}; traces={}
times=torch.linspace(1,0,41)
with torch.no_grad():
    for class_id in [0,1]:
        particle=initial.clone(); labels=torch.full((len(initial),),class_id,dtype=torch.long)
        trajectory=[particle[0].clone()]
        for current,next_t in zip(times[:-1],times[1:]):
            particle=particle+(next_t-current)*field(particle,current.expand(len(particle),1),labels)
            trajectory.append(particle[0].clone())
        generated[class_id]=particle; traces[class_id]=torch.stack(trajectory)
fig,axes=plt.subplots(1,2,figsize=(10,4))
axes[0].scatter(initial[:,0],initial[:,1],s=6,alpha=.3,label='initial noise')
for class_id,points in generated.items():
    axes[0].scatter(points[:,0],points[:,1],s=6,alpha=.5,label=f'class {class_id}')
    axes[1].plot(traces[class_id][:,0],traces[class_id][:,1],'.-',label=f'class {class_id}')
    print('class',class_id,'mean:',points.mean(0).tolist(),'target:',centers[class_id].tolist(),'std:',points.std(0).tolist())
for ax in axes: ax.set_aspect('equal'); ax.legend()
axes[0].set_title('Generated conditional distributions'); axes[1].set_title('Same noise, different conditions')
plt.tight_layout(); plt.show()


## Часть E. Что точно делает Kandinsky 5.0 inference

`get_velocity` → DiT(t×1000) → optional CFG. `generate` → shifted timesteps → visual conditioning → `img += timestep_diff * pred_velocity`. После этого latent делится на VAE scaling factor и декодируется. **Flow Matching не запускается заново при inference:** применяем обученное поле.

Для 5s distill checkpoint config содержит 16 steps и guidance=1.0: conditional velocity без второй uncond ветви. `scheduler_scale` влияет на time grid, а не на архитектуру. Ниже проверяем endpoint/monotonicity и печатаем именно source update. YAML scale=5 и pipeline default=10 расходятся; курс передаёт scale явно.


In [ ]:
from omegaconf import OmegaConf
cfg=OmegaConf.load(SOURCE/'configs/k5_lite_t2v_5s_distil_sd.yaml')
grid=shifted_schedule(cfg.model.num_steps,cfg.metrics.scheduler_scale)
assert grid[0]==1 and grid[-1]==0 and torch.all(grid.diff()<0)
print('steps:',cfg.model.num_steps,'guidance:',cfg.model.guidance_weight,'scale:',cfg.metrics.scheduler_scale)
print('first timesteps:',grid[:5],'sum dt:',grid.diff().sum().item())
source_excerpt('kandinsky/generation_utils.py','pred_velocity = dit(',15)
source_excerpt('kandinsky/generation_utils.py','img[..., :pred_velocity.shape[-1]] +=',4)


### E2. Дистилляция: почему 16 шагов — отдельные обученные веса

Distilled weights учат student эффективно воспроизводить сокращённую trajectory. На inference архитектура всё равно выдаёт velocity и sampler делает Euler updates. Один только аргумент `steps=16` не превращает SFT weights в distilled model. Снижение NFE не гарантирует меньший размер весов или latent activation peak.

Технический отчёт описывает CFG и trajectory distillation, затем post-training; этот notebook не воспроизводит полный distillation trainer. Прямой simple CFM loss выше объясняет foundation training, а не все цели distill checkpoint. [Kandinsky report, training stages](https://arxiv.org/html/2511.14993v1#S6).

### E3. Объединяем VAE и FM теорию

| Этап | Вход | Цель | Какие веса меняются |
|---|---|---|---|
| VAE training | pixels x | reconstruction NLL + KL; у real codecs возможны дополнительные terms | VAE encoder/decoder |
| DiT FM training | scaled VAE data latent, noise, t, text | velocity regression | DiT; external codec/encoders frozen в учебной схеме |
| Distillation | teacher trajectories / conditions | отдельная student objective | Student DiT |
| Generation | noise + text | forward + ODE integration, без training loss | Никакие |

VAE не «убирает Gaussian noise DiT» итерациями: он один раз кодирует/декодирует. DiT не вычисляет ELBO encoder: он учится transport в уже определённом latent space.

**Задания:**

1. Измените time convention на s=1−t, измените target и sampling direction вместе, сравните результаты.
2. Попробуйте убрать time из ToyVelocity; объясните, почему одна стрелка в x может быть недостаточна.
3. Уберите class conditioning и сравните с двумя conditional clouds.
4. Замерьте Euler 8/16/40 steps при одинаковом noise; считайте NFE отдельно от training iterations.
5. В trained tiny DiT сохраните промежуточные latents и оцените distance до known fixed data latent; сравните с random weights.
6. Проследите source: frozen Qwen → trainable LTF → cross-attention → velocity → Euler → frozen decoder.

Итог проверки: вы должны суметь объяснить каждую переменную в Euler строке upstream и воспроизвести forward через отдельные DiT stages без скрытого trainer.


### Проверьте понимание

Не меняйте сразу несколько параметров. Сначала сформулируйте гипотезу, затем измените один параметр и сравните результат с исходным. Запишите: что было входом, что стало выходом, какие размеры изменились и почему.

Учебные модели со случайными весами показывают **механизм**, но не качество Kandinsky. Тяжёлые эксперименты запускаются только при включении соответствующего флага; пропуск этих ячеек не означает успешный запуск настоящих весов.
